In [ ]:

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI

from langchain_community.utilities import SQLDatabase

from langchain_community.agent_toolkits import (
    SQLDatabaseToolkit
)

from langchain.agents import create_agent

In [ ]:
# Configuration

DB_FILE = "data/customer_analytics.db"

MODEL_NAME = "gpt-4o-mini"

In [ ]:
# Connect to SQLite database
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_FILE}"
)

# Create LLM
llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0
)


In [10]:
#Create SQL Toolkit
toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)


# Get tools provided by SQLDatabaseToolkit
tools = toolkit.get_tools()


# 7. Display available tools
print("\n" + "=" * 70)
print("AVAILABLE SQL TOOLS")
print("=" * 70)

for tool in tools:

    print(f"\nTool: {tool.name}")
    print(f"Description: {tool.description}")



AVAILABLE SQL TOOLS

Tool: sql_db_query
Description: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

Tool: sql_db_schema
Description: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

Tool: sql_db_list_tables
Description: Input is an empty string, output is a comma-separated list of tables in the database.

Tool: sql_db_query_checker
Description: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!


In [ ]:
SYSTEM_PROMPT = """
You are an expert SQL database agent.

Your job is to answer the user's question by
exploring the SQLite database and executing SQL.

==================================================
MANDATORY DATABASE EXPLORATION
==================================================

Before generating ANY SQL query, you MUST:

1. Call sql_db_list_tables.

2. From the list of tables, identify ALL tables
   that could contain information relevant to
   the question.

3. Call sql_db_schema for ALL potentially relevant
   tables.

4. Inspect the returned columns and relationships.

5. Determine which columns are needed to answer
   the question.

6. Determine how the relevant tables should be
   joined.

7. Only AFTER completing these steps, generate SQL.

Do NOT generate SQL before inspecting the schema
of all potentially relevant tables.

==================================================
IMPORTANT SEMANTIC RULE
==================================================

Do not assume that one table contains all the
information required by the question.

Break the question into concepts.

For example:

"Which customers churned last month after filing
more than two support tickets?"

contains:

Customer
    -> customers

Churned
    -> subscriptions

Last month
    -> subscriptions.end_date

More than two support tickets
    -> support_tickets

Therefore, subscriptions AND support_tickets
must be considered before generating SQL.

==================================================
SQL RULES
==================================================

- Never invent tables.
- Never invent columns.
- Only use columns discovered through sql_db_schema.
- Generate SELECT queries only.
- Never modify database data.
- Use explicit JOIN conditions.
- Be careful about duplicate rows.
- Use COUNT(DISTINCT ...) when required.
- Always use sql_db_query_checker before
  sql_db_query.
- If SQL execution fails, inspect the error,
  correct the SQL, and retry.
- Do not answer until the SQL query has actually
  been executed successfully.

==================================================
FINAL ANSWER
==================================================

Use only the SQL query result to answer the user.

Do not expose internal reasoning.

Return a concise natural-language answer.
"""

In [ ]:
# Create Agent
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)


In [ ]:
#  Helper functions for displaying agent steps

def print_agent_message(message):

    message_type = getattr(message, "type", None)

    # ========================================================
    # AI MESSAGE
    # ========================================================

    if message_type == "ai":

        # Tool calls
        if getattr(message, "tool_calls", None):

            for tool_call in message.tool_calls:

                print("\n" + "-" * 70)
                print("🔧 AGENT TOOL CALL")
                print("-" * 70)

                print(
                    f"Tool: {tool_call['name']}"
                )

                print(
                    "Arguments:"
                )

                print(
                    tool_call["args"]
                )

        # Normal AI response
        elif message.content:

            print("\n" + "-" * 70)
            print("🤖 AGENT")
            print("-" * 70)

            print(message.content)


    elif message_type == "tool":

        print("\n" + "-" * 70)
        print("📊 TOOL RESULT")
        print("-" * 70)

        tool_name = getattr(message, "name", "")

        print(f"Tool: {tool_name}")

        print(message.content)




In [ ]:
# Run SQL Agent
def run_sql_agent(question):

    print("\nStarting SQL Agent...")

    final_answer = None

    for event in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        stream_mode="updates"
    ):

        for node_name, node_data in event.items():

            messages = node_data.get(
                "messages",
                []
            )

            for message in messages:

                print_agent_message(
                    message
                )

                if (
                    getattr(message, "type", None) == "ai"
                    and isinstance(
                        message.content,
                        str
                    )
                    and message.content.strip()
                    and not getattr(
                        message,
                        "tool_calls",
                        None
                    )
                ):
                    final_answer = message.content


    print("\n")
    print("=" * 70)
    print("FINAL ANSWER")
    print("=" * 70)

    print(final_answer)

    return final_answer


In [11]:

question = """Which customers churned in July 2026 and had filed more than two support tickets?"""
_ = run_sql_agent(question)


Starting SQL Agent...

----------------------------------------------------------------------
🔧 AGENT TOOL CALL
----------------------------------------------------------------------
Tool: sql_db_list_tables
Arguments:
{}

----------------------------------------------------------------------
📊 TOOL RESULT
----------------------------------------------------------------------
Tool: sql_db_list_tables
categories, customers, order_items, orders, payments, products, subscriptions, support_tickets

----------------------------------------------------------------------
🔧 AGENT TOOL CALL
----------------------------------------------------------------------
Tool: sql_db_schema
Arguments:
{'table_names': 'customers'}

----------------------------------------------------------------------
🔧 AGENT TOOL CALL
----------------------------------------------------------------------
Tool: sql_db_schema
Arguments:
{'table_names': 'subscriptions'}

-----------------------------------------------------